# Kinthread final selection (Kaggle, Save & Run All)
Inputs: private dataset `kinthread-real-questions`, r8 / r8p notebook outputs. Picks the checkpoint, trains the verifier, picks the reading setting, all on the full real dev; then scores the locked test once for the pick and for r7 (baseline, no selection on test). Writes FINAL.json.

In [ ]:
import os, glob, hashlib, subprocess, json
REPO = '/kaggle/temp/Kinthread-llm-model'
os.makedirs('/kaggle/temp', exist_ok=True)
if not os.path.exists(REPO):
    !git clone --depth 1 -b v9-reader https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}
%cd {REPO}
tgz = glob.glob('/kaggle/input/**/real_questions_full.tgz', recursive=True)
assert tgz, 'attach the private dataset with real_questions_full.tgz'
!rm -rf data/real_questions && tar -xzf {tgz[0]}
print(len(glob.glob('data/real_questions/*.jsonl')), 'question files')
if not os.path.exists('runs/v10-r7/best.pt'):
    !cat runs/v10-r7/best.pt.part-* > runs/v10-r7/best.pt
assert hashlib.sha256(open('runs/v10-r7/best.pt', 'rb').read()).hexdigest() == open('runs/v10-r7/best.pt.sha256').read().strip()
CKPTS = {'r7': 'runs/v10-r7/best.pt'}
for p in glob.glob('/kaggle/input/**/best.pt', recursive=True):
    CKPTS[p.split('/runs/')[-1].replace('/best.pt', '').replace('v10-', '') + '@' + p.split('/')[3]] = p
print(CKPTS)

In [ ]:
import re, subprocess, json
W = '/kaggle/working'
def dev_fair(ck, flags, tag, split='dev'):
    out = f'{W}/{split}_{tag}.json'
    subprocess.run(f'python -m eval.run_real --ckpt {ck} --split {split} --adaptive 1.0 --hybrid {flags} --out {out}', shell=True)
    s = json.load(open(out))['score']['splits']
    get = lambda k: float(re.search(r'= ([\d.]+)%', s.get(k, '= 0%')).group(1))
    r = {'fair': get(f'{split} fair'), 'verified': get(f'{split} verified'), 'raw': s}
    print(tag, split, r['raw'], flush=True)
    return r
base = {n: dev_fair(c, '', n.replace('@', '_') + '_base') for n, c in CKPTS.items()}
BEST = max(base, key=lambda n: base[n]['fair'])
print('best checkpoint on dev:', BEST, base[BEST])

In [ ]:
!python train_verifier.py --ckpt {CKPTS[BEST]} --n 1500 --out {W}/verifier.json
V = f'{W}/verifier.json'
CONFIGS = {'base': '', 'cover': '--cover', 'para2': '--para 2', 'split': '--split_search', 'verifier': f'--verifier {V}',
           'k16': '--k 16', 'ver+cover+split': f'--verifier {V} --cover --split_search',
           'all': f'--verifier {V} --cover --split_search --para 2 --k 16'}
sweep = {'base': base[BEST]}
for cfg, flags in CONFIGS.items():
    if cfg != 'base':
        sweep[cfg] = dev_fair(CKPTS[BEST], flags, f'{BEST.replace("@", "_")}_{cfg}')
CFG = max(sweep, key=lambda c: sweep[c]['fair'])
print('best setting on dev:', CFG, sweep[CFG])

In [ ]:
final = dev_fair(CKPTS[BEST], CONFIGS[CFG], 'FINAL', split='test')   # locked test: once, for the dev pick
r7 = dev_fair(CKPTS['r7'], '', 'r7_baseline', split='test')      # baseline on the same test, its own default setting
rep = {'checkpoint': BEST, 'setting': CFG, 'flags': CONFIGS[CFG], 'dev_base': base, 'dev_sweep': sweep,
       'locked_test': final, 'r7_locked_test': r7,
       'improvement_over_r7_fair': round(final['fair'] - r7['fair'], 1), 'improvement_over_r7_verified': round(final['verified'] - r7['verified'], 1)}
json.dump(rep, open(f'{W}/FINAL.json', 'w'), indent=1, default=str)
print(json.dumps({k: v for k, v in rep.items() if k not in ('dev_base', 'dev_sweep')}, indent=1, default=str))